In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              matthews_corrcoef, cohen_kappa_score, log_loss, 
                              roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


# Prediction Models

This notebook tests whether the statistically significant moderation effect (confirmed in Notebook 07) can be exploited for practical daily stock direction prediction. Three models are evaluated: logistic regression, LSTM, and XGBoost. All use walk-forward validation to prevent look-ahead bias.

In [2]:
df = pd.read_csv(r"..\..\Data\Main\modelling_dataset.csv")
df['date'] = pd.to_datetime(df['date'])
model_df = df.dropna(subset=['volatility_20d', 'next_day_return']).copy()
model_df = model_df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"Shape: {model_df.shape}")
print(f"Tickers: {model_df['ticker'].nunique()}")
print(f"Date range: {model_df['date'].min().date()} to {model_df['date'].max().date()}")

Shape: (578903, 15)
Tickers: 523
Date range: 2020-01-31 to 2025-12-30


## Define Evaluation Function and Features

In [3]:
def evaluate_predictions(y_true, y_pred, y_prob=None):
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Cohen Kappa': cohen_kappa_score(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            metrics['AUC-ROC'] = roc_auc_score(y_true, y_prob)
            metrics['Log Loss'] = log_loss(y_true, y_prob)
        except:
            metrics['AUC-ROC'] = np.nan
            metrics['Log Loss'] = np.nan
    return metrics

features = ['sentiment', 'volatility_20d', 'sent_x_vol']
target = 'target'

print(f"Features: {features}")
print(f"Target: {target}")
print(f"Evaluation function defined.")

Features: ['sentiment', 'volatility_20d', 'sent_x_vol']
Target: target
Evaluation function defined.


## Walk-Forward Validation Setup

The initial training period is 2020–2021, selected because it captures the two most contrasting market environments in the dataset: 2020 contains the highest volatility (COVID-19 crash, mean volatility 0.034) and 2021 contains the lowest (mean volatility 0.019). The training period is 61.9% volatile and 38.1% calm — not perfectly balanced, but both regimes have substantial representation (102,168 volatile and 62,982 calm observations).

Predictions are made monthly from January 2022 through December 2025 (48 months). The training window expands each month — the model always trains on all available past data before predicting the next month.

In [4]:
train_init = model_df[model_df['date'] < '2022-01-01']
test_period = model_df[model_df['date'] >= '2022-01-01']

print(f"=== Training Period (2020-2021) ===")
print(f"Rows: {len(train_init):,}")
print(f"Tickers: {train_init['ticker'].nunique()}")
print(f"Volatile days: {(train_init['volatile_market'] == 1).sum():,} ({(train_init['volatile_market'] == 1).mean()*100:.1f}%)")
print(f"Calm days: {(train_init['volatile_market'] == 0).sum():,} ({(train_init['volatile_market'] == 0).mean()*100:.1f}%)")
print(f"Target balance: {train_init['target'].mean()*100:.1f}% up / {(1-train_init['target'].mean())*100:.1f}% down")

print(f"\n=== Test Period (2022-2025) ===")
print(f"Rows: {len(test_period):,}")
print(f"Tickers: {test_period['ticker'].nunique()}")
print(f"Volatile days: {(test_period['volatile_market'] == 1).sum():,} ({(test_period['volatile_market'] == 1).mean()*100:.1f}%)")
print(f"Calm days: {(test_period['volatile_market'] == 0).sum():,} ({(test_period['volatile_market'] == 0).mean()*100:.1f}%)")
print(f"Target balance: {test_period['target'].mean()*100:.1f}% up / {(1-test_period['target'].mean())*100:.1f}% down")

model_df['year_month'] = model_df['date'].dt.to_period('M')
all_months = sorted(model_df[model_df['date'] >= '2022-01-01']['year_month'].unique())
print(f"\nPrediction months: {len(all_months)} (from {all_months[0]} to {all_months[-1]})")

=== Training Period (2020-2021) ===
Rows: 165,150
Tickers: 472
Volatile days: 102,168 (61.9%)
Calm days: 62,982 (38.1%)
Target balance: 52.7% up / 47.3% down

=== Test Period (2022-2025) ===
Rows: 413,753
Tickers: 520
Volatile days: 188,007 (45.4%)
Calm days: 225,746 (54.6%)
Target balance: 51.4% up / 48.6% down

Prediction months: 48 (from 2022-01 to 2025-12)


## Model 1: Logistic Regression Walk-Forward

Logistic regression with balanced class weights to prevent majority-class prediction bias. Class weights penalise the model more heavily for missing "down" days, forcing it to make genuine predictions rather than defaulting to always predicting "up."

In [5]:
lr_true = []
lr_pred = []
lr_prob = []
lr_volatile = []

print("Running Logistic Regression walk-forward (48 months)...\n")

for i, month in enumerate(all_months):
    train = model_df[model_df['year_month'] < month]
    test = model_df[model_df['year_month'] == month]
    
    X_train = train[features]
    y_train = train[target]
    X_test = test[features]
    y_test = test[target]
    
    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]
    
    lr_true.extend(y_test.values)
    lr_pred.extend(y_pred)
    lr_prob.extend(y_prob)
    lr_volatile.extend(test['volatile_market'].values)
    
    if (i + 1) % 12 == 0:
        print(f"  Done {i+1}/{len(all_months)} months")

lr_true = np.array(lr_true)
lr_pred = np.array(lr_pred)
lr_prob = np.array(lr_prob)
lr_volatile = np.array(lr_volatile)

print(f"\nTotal predictions: {len(lr_true):,}")
print(f"Predicted up: {lr_pred.sum():,} ({lr_pred.mean()*100:.1f}%)")
print(f"Predicted down: {(1-lr_pred).sum():,.0f} ({(1-lr_pred.mean())*100:.1f}%)")

Running Logistic Regression walk-forward (48 months)...

  Done 12/48 months
  Done 24/48 months
  Done 36/48 months
  Done 48/48 months

Total predictions: 413,753
Predicted up: 224,598 (54.3%)
Predicted down: 189,155 (45.7%)


In [6]:
# Overall metrics
lr_metrics = evaluate_predictions(lr_true, lr_pred, lr_prob)

# Volatile vs calm
vol_mask = lr_volatile == 1
calm_mask = lr_volatile == 0
lr_vol_metrics = evaluate_predictions(lr_true[vol_mask], lr_pred[vol_mask], lr_prob[vol_mask])
lr_calm_metrics = evaluate_predictions(lr_true[calm_mask], lr_pred[calm_mask], lr_prob[calm_mask])

print("=== Logistic Regression: Full Metrics ===\n")
print(f"{'Metric':<15} {'Overall':>10} {'Volatile':>10} {'Calm':>10}")
print("-" * 48)
for name in lr_metrics:
    print(f"{name:<15} {lr_metrics[name]:>10.4f} {lr_vol_metrics[name]:>10.4f} {lr_calm_metrics[name]:>10.4f}")

print(f"\nBaseline accuracy: {model_df[model_df['date'] >= '2022-01-01']['target'].mean():.4f}")

print(f"\n=== Confusion Matrix ===")
cm = confusion_matrix(lr_true, lr_pred)
print(f"                 Predicted Down  Predicted Up")
print(f"  Actual Down    {cm[0][0]:>12,}  {cm[0][1]:>12,}")
print(f"  Actual Up      {cm[1][0]:>12,}  {cm[1][1]:>12,}")

=== Logistic Regression: Full Metrics ===

Metric             Overall   Volatile       Calm
------------------------------------------------
Accuracy            0.5038     0.5043     0.5034
Precision           0.5165     0.5242     0.5105
Recall              0.5454     0.5319     0.5569
F1 Score            0.5306     0.5280     0.5327
MCC                 0.0053     0.0061     0.0051
Cohen Kappa         0.0052     0.0061     0.0051
AUC-ROC             0.5014     0.5001     0.5026
Log Loss            0.6932     0.6931     0.6932

Baseline accuracy: 0.5141

=== Confusion Matrix ===
                 Predicted Down  Predicted Up
  Actual Down          92,448       108,587
  Actual Up            96,707       116,011


## Logistic Regression Interpretation

The logistic regression achieves 50.4% accuracy overall, below the 51.4% baseline. MCC (0.005) and AUC-ROC (0.501) are effectively zero, indicating no meaningful predictive ability. The confusion matrix shows the model is making genuine balanced predictions (54.3% up / 45.7% down) rather than defaulting to one class — it is actively trying to predict but the signal is too weak.

The volatile vs calm accuracy gap is negligible (50.4% vs 50.3%). Unlike the interaction term test in Notebook 07 which found a strong statistical relationship, the logistic regression cannot translate this into practical daily prediction. This is consistent with the sign-flipping finding — the model learns one direction from training data but the relationship reverses in the test period.

## Model 2: XGBoost Walk-Forward

XGBoost is a tree-based machine learning model that can capture non-linear relationships between features. Unlike logistic regression which assumes a linear relationship, XGBoost can learn complex patterns like "sentiment matters when volatility is above a certain threshold." Per-stock models are used because the sentiment-return relationship varies across individual stocks. Class weights are applied to prevent majority-class prediction bias.

### Design Decisions

**Per-stock vs pooled models:** Per-stock models are used because the EDA showed that mean sentiment, volatility, and sector characteristics vary substantially across stocks. A pooled model would learn an average pattern that may not apply to any individual stock. Each stock gets its own model trained on its own history.

**Class weighting:** Balanced class weights are applied using the scale_pos_weight parameter to prevent the model from defaulting to always predicting "up." This was identified as a problem in initial experiments where unweighted models predicted the majority class almost exclusively.

**Monthly walk-forward:** The model retrains every month with an expanding training window. Monthly retraining is chosen  because it provides finer-grained evaluation across 48 test periods.

**Minimum data requirements:** A stock must have at least 100 training days and 5 test days in a given month to be included. 100 training days ensures the model has sufficient history to learn from. 5 test days prevents unreliable accuracy scores from very few predictions.

## Verify data availability

In [7]:
tickers = model_df['ticker'].unique()

# Pre-compute counts per ticker per month
ticker_month_counts = model_df.groupby(['ticker', 'year_month']).size().reset_index(name='count')

# For each ticker, find first month with 100+ cumulative training days
included = 0
excluded = 0
tickers_included = set()

for t in tickers:
    t_data = ticker_month_counts[ticker_month_counts['ticker'] == t].sort_values('year_month')
    t_data['cumcount'] = t_data['count'].cumsum()
    
    for _, row in t_data.iterrows():
        if row['year_month'] < all_months[0]:
            continue
        
        train_size = t_data[t_data['year_month'] < row['year_month']]['count'].sum()
        
        if train_size >= 100 and row['count'] >= 5:
            included += row['count']
            tickers_included.add(t)
        else:
            excluded += row['count']

total = included + excluded
print(f"Predictions included: {included:,} ({included/total*100:.1f}%)")
print(f"Predictions excluded: {excluded:,} ({excluded/total*100:.1f}%)")
print(f"Tickers included (at least once): {len(tickers_included)} out of {len(tickers)}")

Predictions included: 407,812 (98.6%)
Predictions excluded: 5,941 (1.4%)
Tickers included (at least once): 519 out of 523


### Hyperparameter Justification

XGBoost hyperparameters must be chosen before seeing results. The following are selected based on standard practices for financial time series:

In [12]:
from xgboost import XGBClassifier

train_only = model_df[model_df['date'] < '2022-01-01'].copy()
tune_tickers = train_only['ticker'].unique()
print(f"Tuning on all {len(tune_tickers)} tickers")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 5],
    'learning_rate': [0.01, 0.1, 0.2],
}

results = []
total = len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['learning_rate'])
count = 0

print(f"Testing {total} combinations using time-series CV on 2020-2021 data...\n")

for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for lr in param_grid['learning_rate']:
            count += 1
            fold_accs = []
            
            for t in tune_tickers:
                t_data = train_only[train_only['ticker'] == t].sort_values('date')
                
                split_idx = int(len(t_data) * 0.7)
                cv_train = t_data.iloc[:split_idx]
                cv_val = t_data.iloc[split_idx:]
                
                if len(cv_train) < 50 or len(cv_val) < 20:
                    continue
                
                xgb = XGBClassifier(
                    n_estimators=n_est, max_depth=depth, learning_rate=lr,
                    random_state=42,
                    scale_pos_weight=max(len(cv_train[cv_train[target]==0]), 1) / max(len(cv_train[cv_train[target]==1]), 1),
                    use_label_encoder=False, eval_metric='logloss', verbosity=0
                )
                xgb.fit(cv_train[features], cv_train[target])
                y_pred = xgb.predict(cv_val[features])
                fold_accs.append(accuracy_score(cv_val[target], y_pred))
            
            mean_acc = np.mean(fold_accs) if fold_accs else 0
            results.append({
                'n_estimators': n_est, 'max_depth': depth, 
                'learning_rate': lr, 'mean_accuracy': mean_acc
            })
            print(f"  {count}/{total}: n_est={n_est}, depth={depth}, lr={lr}, acc={mean_acc:.4f}")

param_df = pd.DataFrame(results).sort_values('mean_accuracy', ascending=False)

print(f"\n=== All Results ===\n")
print(f"{'n_est':>6} {'depth':>6} {'lr':>8} {'accuracy':>10}")
print("-" * 35)
for _, row in param_df.iterrows():
    print(f"{int(row['n_estimators']):>6} {int(row['max_depth']):>6} {row['learning_rate']:>8.2f} {row['mean_accuracy']:>10.4f}")

best = param_df.iloc[0]
print(f"\n=== Best Parameters ===")
print(f"n_estimators: {int(best['n_estimators'])}")
print(f"max_depth: {int(best['max_depth'])}")
print(f"learning_rate: {best['learning_rate']}")
print(f"Validation accuracy: {best['mean_accuracy']:.4f}")

BEST_N_EST = int(best['n_estimators'])
BEST_DEPTH = int(best['max_depth'])
BEST_LR = best['learning_rate']

Tuning on all 472 tickers
Testing 27 combinations using time-series CV on 2020-2021 data...

  1/27: n_est=50, depth=2, lr=0.01, acc=0.5022
  2/27: n_est=50, depth=2, lr=0.1, acc=0.4981
  3/27: n_est=50, depth=2, lr=0.2, acc=0.4975
  4/27: n_est=50, depth=3, lr=0.01, acc=0.4993
  5/27: n_est=50, depth=3, lr=0.1, acc=0.5005
  6/27: n_est=50, depth=3, lr=0.2, acc=0.4995
  7/27: n_est=50, depth=5, lr=0.01, acc=0.4995
  8/27: n_est=50, depth=5, lr=0.1, acc=0.4988
  9/27: n_est=50, depth=5, lr=0.2, acc=0.4984
  10/27: n_est=100, depth=2, lr=0.01, acc=0.5007
  11/27: n_est=100, depth=2, lr=0.1, acc=0.4991
  12/27: n_est=100, depth=2, lr=0.2, acc=0.4990
  13/27: n_est=100, depth=3, lr=0.01, acc=0.5014
  14/27: n_est=100, depth=3, lr=0.1, acc=0.5005
  15/27: n_est=100, depth=3, lr=0.2, acc=0.4991
  16/27: n_est=100, depth=5, lr=0.01, acc=0.5018
  17/27: n_est=100, depth=5, lr=0.1, acc=0.4985
  18/27: n_est=100, depth=5, lr=0.2, acc=0.4998
  19/27: n_est=200, depth=2, lr=0.01, acc=0.5001
  20/2

### Hyperparameter Tuning Results

All 27 combinations produced accuracy between 49.75% and 50.22% — a range of less than 0.5%. This indicates the results are robust to hyperparameter choices and not dependent on any specific configuration. The best-performing combination (n_estimators=50, max_depth=2, learning_rate=0.01) is the simplest tested configuration, consistent with the principle that weaker signals favour simpler models to avoid overfitting.

In [13]:
xgb_true = []
xgb_pred = []
xgb_prob = []
xgb_volatile = []

start_time = time.time()
print(f"Running XGBoost per-stock walk-forward")
print(f"Parameters: n_estimators={BEST_N_EST}, max_depth={BEST_DEPTH}, learning_rate={BEST_LR}\n")

for mi, month in enumerate(all_months):
    train = model_df[model_df['year_month'] < month]
    test = model_df[model_df['year_month'] == month]
    
    for t in tickers:
        train_t = train[train['ticker'] == t]
        test_t = test[test['ticker'] == t]
        
        if len(train_t) < 100 or len(test_t) < 5:
            continue
        
        xgb = XGBClassifier(
            n_estimators=BEST_N_EST, max_depth=BEST_DEPTH, learning_rate=BEST_LR,
            random_state=42,
            scale_pos_weight=max(len(train_t[train_t[target]==0]), 1) / max(len(train_t[train_t[target]==1]), 1),
            use_label_encoder=False, eval_metric='logloss', verbosity=0
        )
        xgb.fit(train_t[features], train_t[target])
        
        y_pred = xgb.predict(test_t[features])
        y_prob = xgb.predict_proba(test_t[features])[:, 1]
        
        xgb_true.extend(test_t[target].values)
        xgb_pred.extend(y_pred)
        xgb_prob.extend(y_prob)
        xgb_volatile.extend(test_t['volatile_market'].values)
    
    if (mi + 1) % 12 == 0:
        elapsed = time.time() - start_time
        print(f"  Done {mi+1}/{len(all_months)} months [{elapsed:.0f}s]")

xgb_true = np.array(xgb_true)
xgb_pred = np.array(xgb_pred)
xgb_prob = np.array(xgb_prob)
xgb_volatile = np.array(xgb_volatile)

elapsed = time.time() - start_time
print(f"\nComplete! {elapsed:.0f}s")
print(f"Total predictions: {len(xgb_true):,}")
print(f"Predicted up: {xgb_pred.sum():,} ({xgb_pred.mean()*100:.1f}%)")
print(f"Predicted down: {len(xgb_pred)-xgb_pred.sum():,} ({(1-xgb_pred.mean())*100:.1f}%)")

Running XGBoost per-stock walk-forward
Parameters: n_estimators=50, max_depth=2, learning_rate=0.01

  Done 12/48 months [184s]
  Done 24/48 months [417s]
  Done 36/48 months [700s]
  Done 48/48 months [1029s]

Complete! 1029s
Total predictions: 407,812
Predicted up: 208,538 (51.1%)
Predicted down: 199,274 (48.9%)


In [14]:
xgb_metrics = evaluate_predictions(xgb_true, xgb_pred, xgb_prob)

vol_mask = xgb_volatile == 1
calm_mask = xgb_volatile == 0
xgb_vol_metrics = evaluate_predictions(xgb_true[vol_mask], xgb_pred[vol_mask], xgb_prob[vol_mask])
xgb_calm_metrics = evaluate_predictions(xgb_true[calm_mask], xgb_pred[calm_mask], xgb_prob[calm_mask])

print("=== XGBoost Per-Stock: Full Metrics ===\n")
print(f"{'Metric':<15} {'Overall':>10} {'Volatile':>10} {'Calm':>10}")
print("-" * 48)
for name in xgb_metrics:
    print(f"{name:<15} {xgb_metrics[name]:>10.4f} {xgb_vol_metrics[name]:>10.4f} {xgb_calm_metrics[name]:>10.4f}")

print(f"\nBaseline accuracy: 0.5141")

print(f"\n=== Confusion Matrix ===")
cm = confusion_matrix(xgb_true, xgb_pred)
print(f"                 Predicted Down  Predicted Up")
print(f"  Actual Down    {cm[0][0]:>12,}  {cm[0][1]:>12,}")
print(f"  Actual Up      {cm[1][0]:>12,}  {cm[1][1]:>12,}")

=== XGBoost Per-Stock: Full Metrics ===

Metric             Overall   Volatile       Calm
------------------------------------------------
Accuracy            0.5008     0.5014     0.5004
Precision           0.5147     0.5217     0.5087
Recall              0.5119     0.5214     0.5037
F1 Score            0.5133     0.5216     0.5062
MCC                 0.0010     0.0009     0.0007
Cohen Kappa         0.0010     0.0009     0.0007
AUC-ROC             0.5003     0.5005     0.4998
Log Loss            0.6943     0.6943     0.6943

Baseline accuracy: 0.5141

=== Confusion Matrix ===
                 Predicted Down  Predicted Up
  Actual Down          96,911       101,202
  Actual Up           102,363       107,336


## XGBoost Interpretation

XGBoost per-stock models achieve 50.1% accuracy overall with MCC of 0.001 — effectively no predictive ability. Despite using non-linear tree-based models trained individually per stock with tuned hyperparameters, the results are no better than logistic regression. The volatile vs calm gap is negligible (50.1% vs 50.0%).

This confirms that the prediction limitation is not due to model choice or linearity assumptions. The same three features (sentiment, volatility, interaction) produce ~50% accuracy whether processed by a linear model (logistic regression) or a non-linear model (XGBoost). The features themselves do not contain sufficient signal for daily direction prediction — consistent with the sign-flipping finding from Notebook 07.

In [17]:
print("=== Model Comparison So Far ===\n")

print("--- Overall ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name, metrics in [
    ('Logistic Regression', lr_metrics),
    ('XGBoost Per-Stock', xgb_metrics),
]:
    print(f"{name:<25} {metrics['Accuracy']:>10.4f} {metrics['F1 Score']:>10.4f} {metrics['MCC']:>10.4f} {metrics['AUC-ROC']:>10.4f}")

print(f"\n--- Volatile Markets ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name, metrics in [
    ('Logistic Regression', lr_vol_metrics),
    ('XGBoost Per-Stock', xgb_vol_metrics),
]:
    print(f"{name:<25} {metrics['Accuracy']:>10.4f} {metrics['F1 Score']:>10.4f} {metrics['MCC']:>10.4f} {metrics['AUC-ROC']:>10.4f}")

print(f"\n--- Calm Markets ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name, metrics in [
    ('Logistic Regression', lr_calm_metrics),
    ('XGBoost Per-Stock', xgb_calm_metrics),
]:
    print(f"{name:<25} {metrics['Accuracy']:>10.4f} {metrics['F1 Score']:>10.4f} {metrics['MCC']:>10.4f} {metrics['AUC-ROC']:>10.4f}")

print(f"\nBaseline accuracy: 0.5141")

=== Model Comparison So Far ===

--- Overall ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
Logistic Regression           0.5038     0.5306     0.0053     0.5014
XGBoost Per-Stock             0.5008     0.5133     0.0010     0.5003

--- Volatile Markets ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
Logistic Regression           0.5043     0.5280     0.0061     0.5001
XGBoost Per-Stock             0.5014     0.5216     0.0009     0.5005

--- Calm Markets ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
Logistic Regression           0.5034     0.5327     0.0051     0.5026
XGBoost Per-Stock             0.5004     0.5062     0.0007     0.4998

Baseline accuracy: 0.5141


## Model 3: LSTM Walk-Forward

LSTM (Long Short-Term Memory) is a deep learning model that reads sequences of past days rather than single-day snapshots. It can detect patterns like "sentiment has been declining for a week" that single-day models miss. Three versions are tested:

- **LSTM-A:** Sentiment sequence only — can sentiment patterns over 20 days predict direction?
- **LSTM-B:** Volatility sequence only — can volatility patterns predict direction?
- **LSTM-C:** Combined — does adding both improve prediction?

Quarterly walk-forward is used. The initial training period is 2020–2021, with predictions from Q1 2022 through Q4 2025 (16 quarters).

### Design Decisions

**Sequence length (20 days):** Matches the 20-day volatility calculation window, ensuring the LSTM sees the same history used to compute volatility. Also represents approximately one trading month.

**Balanced loss:** BCEWithLogitsLoss with pos_weight prevents majority-class prediction bias, the same issue addressed with class_weight='balanced' in logistic regression and scale_pos_weight in XGBoost.

**Standardisation:** Features are scaled using StandardScaler fitted on training data only, ensuring no information leaks from the test period.

### LSTM Architecture

The model consists of:
- **LSTM layer:** Reads the 20-day sequence one day at a time, building a summary of the temporal pattern
- **Output layer:** Converts the LSTM summary into a single prediction (probability of stock going up)

Hyperparameters (hidden size, epochs, sequence length) are tuned using a time-series validation split within the training period only (train on 2020 to June 2021, validate on July to December 2021). This ensures no test-period information influences model configuration.

### Hyperparameter Tuning

Tuning is performed using the combined feature set (sentiment + volatility) since this is the most relevant configuration for the research question. The best parameters are then applied to all three feature sets for fair comparison.

The grid search tests:
- Hidden size: 32, 64, 128 (model memory capacity)
- Epochs: 5, 10, 15 (number of training passes)
- Sequence length: 10, 20, 30 (days of history the model sees)

This produces 27 combinations tested on training-period data only.

In [19]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        out = self.fc(last_output)
        return out

def create_sequences(data, features, seq_length, target_col='target'):
    all_X, all_y, all_dates, all_volatile = [], [], [], []
    
    for t in data['ticker'].unique():
        ticker_data = data[data['ticker'] == t].sort_values('date')
        feat_values = ticker_data[features].values
        target_values = ticker_data[target_col].values
        dates = ticker_data['date'].values
        volatile = ticker_data['volatile_market'].values
        
        for j in range(seq_length, len(ticker_data)):
            all_X.append(feat_values[j - seq_length:j])
            all_y.append(target_values[j])
            all_dates.append(dates[j])
            all_volatile.append(volatile[j])
    
    return np.array(all_X), np.array(all_y), np.array(all_dates), np.array(all_volatile)

# Tuning split: train on 2020 to Jun 2021, validate on Jul to Dec 2021
tune_features = ['sentiment', 'volatility_20d']
train_period = model_df[model_df['date'] < '2021-07-01']
val_period = model_df[(model_df['date'] >= '2021-07-01') & (model_df['date'] < '2022-01-01')]

print(f"Tuning split:")
print(f"  Train: {train_period['date'].min().date()} to {train_period['date'].max().date()} ({len(train_period):,} rows)")
print(f"  Validate: {val_period['date'].min().date()} to {val_period['date'].max().date()} ({len(val_period):,} rows)")

param_grid = {
    'hidden_size': [32, 64, 128],
    'epochs': [5, 10, 15],
    'seq_length': [10, 20, 30],
}

total = 27
count = 0

print(f"\nTesting {total} combinations with balanced loss...\n")
print(f"{'#':<4} {'Hidden':>7} {'Epochs':>7} {'SeqLen':>7} {'Accuracy':>10} {'MCC':>10} {'Pred Up%':>10} {'Time':>7}")
print("-" * 65)

grid_results = []

for hs in param_grid['hidden_size']:
    for ep in param_grid['epochs']:
        for sl in param_grid['seq_length']:
            count += 1
            start = time.time()
            
            X_train, y_train, _, _ = create_sequences(train_period, tune_features, sl)
            X_val, y_val, _, vol_val = create_sequences(val_period, tune_features, sl)
            
            if len(X_val) == 0:
                continue
            
            scaler = StandardScaler()
            n_tr, seql, nf = X_train.shape
            X_train_s = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(n_tr, seql, nf)
            X_val_s = scaler.transform(X_val.reshape(-1, nf)).reshape(len(X_val), seql, nf)
            
            n_pos = y_train.sum()
            n_neg = len(y_train) - n_pos
            pos_weight = torch.FloatTensor([n_neg / n_pos]).to(device)
            
            model = LSTMModel(input_size=len(tune_features), hidden_size=hs, num_layers=1).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            X_tr_t = torch.FloatTensor(X_train_s).to(device)
            y_tr_t = torch.FloatTensor(y_train).to(device)
            loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=1024, shuffle=True)
            
            model.train()
            for epoch in range(ep):
                for batch_X, batch_y in loader:
                    optimizer.zero_grad()
                    output = model(batch_X).squeeze()
                    loss = criterion(output, batch_y)
                    loss.backward()
                    optimizer.step()
            
            model.eval()
            X_val_t = torch.FloatTensor(X_val_s).to(device)
            with torch.no_grad():
                logits = model(X_val_t).squeeze().cpu().numpy()
                probs = 1 / (1 + np.exp(-logits))
            
            preds = (probs > 0.5).astype(int)
            acc = accuracy_score(y_val, preds)
            mcc = matthews_corrcoef(y_val, preds)
            
            elapsed = time.time() - start
            
            grid_results.append({
                'hidden_size': hs, 'epochs': ep, 'seq_length': sl,
                'accuracy': acc, 'mcc': mcc, 'pred_up_pct': preds.mean()
            })
            
            print(f"{count:<4} {hs:>7} {ep:>7} {sl:>7} {acc:>10.4f} {mcc:>10.4f} {preds.mean():>9.1%} {elapsed:>6.1f}s")

grid_df = pd.DataFrame(grid_results).sort_values('accuracy', ascending=False)

print(f"\n=== Best Parameters ===")
best = grid_df.iloc[0]
print(f"Hidden size: {int(best['hidden_size'])}")
print(f"Epochs: {int(best['epochs'])}")
print(f"Sequence length: {int(best['seq_length'])}")
print(f"Validation accuracy: {best['accuracy']:.4f}")
print(f"Validation MCC: {best['mcc']:.4f}")

BEST_HIDDEN = int(best['hidden_size'])
BEST_EPOCHS = int(best['epochs'])
BEST_SEQ = int(best['seq_length'])

print(f"\nAccuracy range: {grid_df['accuracy'].min():.4f} to {grid_df['accuracy'].max():.4f} (spread: {grid_df['accuracy'].max()-grid_df['accuracy'].min():.4f})")

Tuning split:
  Train: 2020-01-31 to 2021-06-29 (116,354 rows)
  Validate: 2021-07-01 to 2021-12-31 (48,796 rows)

Testing 27 combinations with balanced loss...

#     Hidden  Epochs  SeqLen   Accuracy        MCC   Pred Up%    Time
-----------------------------------------------------------------
1         32       5      10     0.4898    -0.0079     39.8%   11.2s
2         32       5      20     0.4758    -0.0190     22.5%   13.3s
3         32       5      30     0.4936    -0.0124     49.6%   15.3s
4         32      10      10     0.4766    -0.0139     21.6%   17.3s
5         32      10      20     0.4955    -0.0144     54.6%   23.0s
6         32      10      30     0.4870    -0.0160     41.1%   26.3s
7         32      15      10     0.5130    -0.0155     80.6%   24.6s
8         32      15      20     0.4814    -0.0085     24.7%   31.3s
9         32      15      30     0.4968    -0.0070     50.4%   37.7s
10        64       5      10     0.4802    -0.0077     23.5%   15.9s
11        64

### Hyperparameter Tuning Results

The tuning results and best parameters are reported above. The accuracy range across all combinations indicates whether the model is sensitive to hyperparameter choices.

In [21]:
# Filter for models with reasonable prediction balance
balanced = grid_df[(grid_df['pred_up_pct'] >= 0.40) & (grid_df['pred_up_pct'] <= 0.60)].copy()

print(f"Models with balanced predictions (40-60% up): {len(balanced)} out of {len(grid_df)}\n")

# Sort by MCC (best discriminative ability) not accuracy
balanced = balanced.sort_values('mcc', ascending=False)

print(f"{'#':<4} {'Hidden':>7} {'Epochs':>7} {'SeqLen':>7} {'Accuracy':>10} {'MCC':>10} {'Pred Up%':>10}")
print("-" * 60)
for i, (_, row) in enumerate(balanced.iterrows()):
    print(f"{i+1:<4} {int(row['hidden_size']):>7} {int(row['epochs']):>7} {int(row['seq_length']):>7} {row['accuracy']:>10.4f} {row['mcc']:>10.4f} {row['pred_up_pct']:>9.1%}")

if len(balanced) > 0:
    best = balanced.iloc[0]
    BEST_HIDDEN = int(best['hidden_size'])
    BEST_EPOCHS = int(best['epochs'])
    BEST_SEQ = int(best['seq_length'])
    
    print(f"\n=== Selected Parameters ===")
    print(f"Hidden size: {BEST_HIDDEN}")
    print(f"Epochs: {BEST_EPOCHS}")
    print(f"Sequence length: {BEST_SEQ}")
    print(f"Validation accuracy: {best['accuracy']:.4f}")
    print(f"Validation MCC: {best['mcc']:.4f}")
    print(f"Prediction balance: {best['pred_up_pct']:.1%} up")
else:
    print("No balanced models found — using default parameters")
    BEST_HIDDEN = 64
    BEST_EPOCHS = 5
    BEST_SEQ = 20

Models with balanced predictions (40-60% up): 9 out of 27

#     Hidden  Epochs  SeqLen   Accuracy        MCC   Pred Up%
------------------------------------------------------------
1         32      15      30     0.4968    -0.0070     50.4%
2         32       5      30     0.4936    -0.0124     49.6%
3         32      10      20     0.4955    -0.0144     54.6%
4        128      15      10     0.4915    -0.0155     48.8%
5         32      10      30     0.4870    -0.0160     41.1%
6         64      10      10     0.4949    -0.0185     56.5%
7        128      15      30     0.4862    -0.0211     44.3%
8         64      15      20     0.4910    -0.0257     56.3%
9        128      10      20     0.4930    -0.0263     59.9%

=== Selected Parameters ===
Hidden size: 32
Epochs: 15
Sequence length: 30
Validation accuracy: 0.4968
Validation MCC: -0.0070
Prediction balance: 50.4% up


### Hyperparameter Selection

Of 27 tested combinations, only 9 produced balanced predictions (40-60% up). Many configurations defaulted to extreme prediction biases despite the balanced loss function, indicating LSTM training instability with this data. The selected parameters (hidden=32, epochs=15, seq_len=30) produced the most balanced predictions with the highest MCC among balanced models. All balanced models showed MCC near zero, consistent with the logistic regression and XGBoost findings.

In [22]:
# Build sequences for all three feature sets using best sequence length
feature_sets_lstm = {
    'A_sentiment_only': ['sentiment'],
    'B_volatility_only': ['volatility_20d'],
    'C_combined': ['sentiment', 'volatility_20d']
}

print(f"Using best parameters: hidden={BEST_HIDDEN}, epochs={BEST_EPOCHS}, seq_len={BEST_SEQ}\n")
print("Building sequences...")

sequences = {}
for name, feats in feature_sets_lstm.items():
    X, y, d, v = create_sequences(model_df, feats, BEST_SEQ)
    sequences[name] = {'X': X, 'y': y, 'dates': d, 'volatile': v}
    print(f"  {name}: {X.shape}")

quarters_start = pd.date_range('2022-01-01', '2025-10-01', freq='QS')
quarters_end = pd.date_range('2022-03-31', '2025-12-31', freq='QE')

print(f"\nQuarters: {len(quarters_start)} (Q1 2022 to Q4 2025)")

# Walk-forward for all three feature sets
lstm_results = {}

for model_name, feats in feature_sets_lstm.items():
    print(f"\n{'='*60}")
    print(f"Running LSTM-{model_name} (features: {feats})")
    print(f"{'='*60}")
    
    X = sequences[model_name]['X']
    y = sequences[model_name]['y']
    dates_arr = pd.to_datetime(sequences[model_name]['dates'])
    volatile = sequences[model_name]['volatile']
    input_size = len(feats)
    
    all_true, all_pred, all_prob, all_vol = [], [], [], []
    start_time = time.time()
    
    for qi, (qs, qe) in enumerate(zip(quarters_start, quarters_end)):
        train_mask = dates_arr < qs
        test_mask = (dates_arr >= qs) & (dates_arr <= qe)
        
        X_train, y_train = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]
        volatile_test = volatile[test_mask]
        
        if len(X_test) == 0:
            continue
        
        n_pos = y_train.sum()
        n_neg = len(y_train) - n_pos
        pos_weight = torch.FloatTensor([n_neg / n_pos]).to(device)
        
        scaler = StandardScaler()
        n_train, sl, nf = X_train.shape
        X_train_scaled = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(n_train, sl, nf)
        X_test_scaled = scaler.transform(X_test.reshape(-1, nf)).reshape(len(X_test), sl, nf)
        
        model = LSTMModel(input_size=input_size, hidden_size=BEST_HIDDEN, num_layers=1).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        X_train_t = torch.FloatTensor(X_train_scaled).to(device)
        y_train_t = torch.FloatTensor(y_train).to(device)
        train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=1024, shuffle=True)
        
        model.train()
        for epoch in range(BEST_EPOCHS):
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                output = model(batch_X).squeeze()
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
        
        model.eval()
        X_test_t = torch.FloatTensor(X_test_scaled).to(device)
        with torch.no_grad():
            logits = model(X_test_t).squeeze().cpu().numpy()
            probs = 1 / (1 + np.exp(-logits))
        
        preds = (probs > 0.5).astype(int)
        
        all_true.extend(y_test)
        all_pred.extend(preds)
        all_prob.extend(probs)
        all_vol.extend(volatile_test)
        
        elapsed = time.time() - start_time
        print(f"  Q{qi+1}: pred_up={preds.sum()}, pred_down={len(preds)-preds.sum()}, total={len(preds)} [{elapsed:.0f}s]")
    
    lstm_results[model_name] = {
        'y_true': np.array(all_true),
        'y_pred': np.array(all_pred),
        'y_prob': np.array(all_prob),
        'volatile': np.array(all_vol)
    }
    
    total_time = time.time() - start_time
    print(f"  Complete: {total_time:.0f}s, predictions: {len(all_true):,}")

print(f"\n=== All LSTM models complete ===")

Using best parameters: hidden=32, epochs=15, seq_len=30

Building sequences...
  A_sentiment_only: (563213, 30, 1)
  B_volatility_only: (563213, 30, 1)
  C_combined: (563213, 30, 2)

Quarters: 16 (Q1 2022 to Q4 2025)

Running LSTM-A_sentiment_only (features: ['sentiment'])
  Q1: pred_up=18698, pred_down=7940, total=26638 [48s]
  Q2: pred_up=10980, pred_down=15069, total=26049 [106s]
  Q3: pred_up=13473, pred_down=12937, total=26410 [173s]
  Q4: pred_up=9838, pred_down=16047, total=25885 [249s]
  Q5: pred_up=11215, pred_down=14742, total=25957 [335s]
  Q6: pred_up=12953, pred_down=12944, total=25897 [428s]
  Q7: pred_up=20002, pred_down=6465, total=26467 [532s]
  Q8: pred_up=12364, pred_down=13465, total=25829 [644s]
  Q9: pred_up=17284, pred_down=8204, total=25488 [777s]
  Q10: pred_up=19602, pred_down=6237, total=25839 [906s]
  Q11: pred_up=20050, pred_down=6451, total=26501 [1059s]
  Q12: pred_up=22541, pred_down=4214, total=26755 [1205s]
  Q13: pred_up=15472, pred_down=8607, total=2

In [23]:
print("=== LSTM Results ===\n")

lstm_all_metrics = {}
for name in feature_sets_lstm:
    r = lstm_results[name]
    lstm_all_metrics[name] = {
        'overall': evaluate_predictions(r['y_true'], r['y_pred'], r['y_prob']),
        'volatile': evaluate_predictions(r['y_true'][r['volatile']==1], r['y_pred'][r['volatile']==1], r['y_prob'][r['volatile']==1]),
        'calm': evaluate_predictions(r['y_true'][r['volatile']==0], r['y_pred'][r['volatile']==0], r['y_prob'][r['volatile']==0])
    }

print("--- Overall ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['overall']
    print(f"LSTM-{name:<20} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\n--- Volatile Markets ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['volatile']
    print(f"LSTM-{name:<20} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\n--- Calm Markets ---")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 68)
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['calm']
    print(f"LSTM-{name:<20} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\nBaseline accuracy: 0.5141")

print(f"\n--- Prediction Balance ---")
for name in feature_sets_lstm:
    r = lstm_results[name]
    print(f"  LSTM-{name}: up={r['y_pred'].sum():,} ({r['y_pred'].mean()*100:.1f}%), down={len(r['y_pred'])-r['y_pred'].sum():,} ({(1-r['y_pred'].mean())*100:.1f}%)")

print(f"\n--- Confusion Matrices ---")
for name in feature_sets_lstm:
    r = lstm_results[name]
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    print(f"\n  LSTM-{name}:")
    print(f"                   Predicted Down  Predicted Up")
    print(f"    Actual Down    {cm[0][0]:>12,}  {cm[0][1]:>12,}")
    print(f"    Actual Up      {cm[1][0]:>12,}  {cm[1][1]:>12,}")

=== LSTM Results ===

--- Overall ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
LSTM-A_sentiment_only         0.5031     0.5685    -0.0016     0.4963
LSTM-B_volatility_only        0.4866     0.4426    -0.0219     0.4867
LSTM-C_combined               0.5040     0.5634     0.0011     0.4990

--- Volatile Markets ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
LSTM-A_sentiment_only         0.5050     0.5551     0.0023     0.5006
LSTM-B_volatility_only        0.4881     0.4722    -0.0195     0.4870
LSTM-C_combined               0.5053     0.5627     0.0014     0.5001

--- Calm Markets ---
Model                       Accuracy         F1        MCC    AUC-ROC
--------------------------------------------------------------------
LSTM-A_sentiment_only         0.5015     0.5790    -0.0030     0.4931
LSTM-B_

In [24]:
print("=" * 90)
print("FINAL MODEL COMPARISON — ALL MODELS")
print("=" * 90)

print("\n--- Overall ---")
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 73)
print(f"{'Logistic Regression':<30} {lr_metrics['Accuracy']:>10.4f} {lr_metrics['F1 Score']:>10.4f} {lr_metrics['MCC']:>10.4f} {lr_metrics['AUC-ROC']:>10.4f}")
print(f"{'XGBoost Per-Stock':<30} {xgb_metrics['Accuracy']:>10.4f} {xgb_metrics['F1 Score']:>10.4f} {xgb_metrics['MCC']:>10.4f} {xgb_metrics['AUC-ROC']:>10.4f}")
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['overall']
    print(f"{'LSTM-' + name:<30} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\n--- Volatile Markets ---")
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 73)
print(f"{'Logistic Regression':<30} {lr_vol_metrics['Accuracy']:>10.4f} {lr_vol_metrics['F1 Score']:>10.4f} {lr_vol_metrics['MCC']:>10.4f} {lr_vol_metrics['AUC-ROC']:>10.4f}")
print(f"{'XGBoost Per-Stock':<30} {xgb_vol_metrics['Accuracy']:>10.4f} {xgb_vol_metrics['F1 Score']:>10.4f} {xgb_vol_metrics['MCC']:>10.4f} {xgb_vol_metrics['AUC-ROC']:>10.4f}")
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['volatile']
    print(f"{'LSTM-' + name:<30} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\n--- Calm Markets ---")
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 73)
print(f"{'Logistic Regression':<30} {lr_calm_metrics['Accuracy']:>10.4f} {lr_calm_metrics['F1 Score']:>10.4f} {lr_calm_metrics['MCC']:>10.4f} {lr_calm_metrics['AUC-ROC']:>10.4f}")
print(f"{'XGBoost Per-Stock':<30} {xgb_calm_metrics['Accuracy']:>10.4f} {xgb_calm_metrics['F1 Score']:>10.4f} {xgb_calm_metrics['MCC']:>10.4f} {xgb_calm_metrics['AUC-ROC']:>10.4f}")
for name in feature_sets_lstm:
    m = lstm_all_metrics[name]['calm']
    print(f"{'LSTM-' + name:<30} {m['Accuracy']:>10.4f} {m['F1 Score']:>10.4f} {m['MCC']:>10.4f} {m['AUC-ROC']:>10.4f}")

print(f"\nBaseline accuracy: 0.5141")

FINAL MODEL COMPARISON — ALL MODELS

--- Overall ---
Model                            Accuracy         F1        MCC    AUC-ROC
-------------------------------------------------------------------------
Logistic Regression                0.5038     0.5306     0.0053     0.5014
XGBoost Per-Stock                  0.5008     0.5133     0.0010     0.5003
LSTM-A_sentiment_only              0.5031     0.5685    -0.0016     0.4963
LSTM-B_volatility_only             0.4866     0.4426    -0.0219     0.4867
LSTM-C_combined                    0.5040     0.5634     0.0011     0.4990

--- Volatile Markets ---
Model                            Accuracy         F1        MCC    AUC-ROC
-------------------------------------------------------------------------
Logistic Regression                0.5043     0.5280     0.0061     0.5001
XGBoost Per-Stock                  0.5014     0.5216     0.0009     0.5005
LSTM-A_sentiment_only              0.5050     0.5551     0.0023     0.5006
LSTM-B_volatility_only 

## Supplementary: LSTM with Price Data

As an additional test, the LSTM is given log_return alongside sentiment and volatility as sequence inputs. This provides the model with actual price movement history — the type of sequential pattern LSTM is specifically designed to detect. The explicit interaction term (sent_x_vol) is excluded because the LSTM can learn non-linear feature interactions implicitly from sequential data.

In [25]:
# LSTM with log_return — run overnight
print("Building sequences with log_return...")
lstm_feats_extended = ['sentiment', 'volatility_20d', 'log_return']

X_ext, y_ext, d_ext, v_ext = create_sequences(model_df, lstm_feats_extended, BEST_SEQ)
dates_ext = pd.to_datetime(d_ext)
print(f"Shape: {X_ext.shape}")

print(f"\nRunning LSTM with {lstm_feats_extended}")
print(f"Parameters: hidden={BEST_HIDDEN}, epochs={BEST_EPOCHS}, seq_len={BEST_SEQ}\n")

all_true, all_pred, all_prob, all_vol = [], [], [], []
start_time = time.time()

for qi, (qs, qe) in enumerate(zip(quarters_start, quarters_end)):
    train_mask = dates_ext < qs
    test_mask = (dates_ext >= qs) & (dates_ext <= qe)
    
    X_train, y_train = X_ext[train_mask], y_ext[train_mask]
    X_test, y_test = X_ext[test_mask], y_ext[test_mask]
    volatile_test = v_ext[test_mask]
    
    if len(X_test) == 0:
        continue
    
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.FloatTensor([n_neg / n_pos]).to(device)
    
    scaler = StandardScaler()
    n_train, sl, nf = X_train.shape
    X_train_scaled = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(n_train, sl, nf)
    X_test_scaled = scaler.transform(X_test.reshape(-1, nf)).reshape(len(X_test), sl, nf)
    
    model = LSTMModel(input_size=len(lstm_feats_extended), hidden_size=BEST_HIDDEN, num_layers=1).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    X_train_t = torch.FloatTensor(X_train_scaled).to(device)
    y_train_t = torch.FloatTensor(y_train).to(device)
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=1024, shuffle=True)
    
    model.train()
    for epoch in range(BEST_EPOCHS):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_X).squeeze()
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
    
    model.eval()
    X_test_t = torch.FloatTensor(X_test_scaled).to(device)
    with torch.no_grad():
        logits = model(X_test_t).squeeze().cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))
    
    preds = (probs > 0.5).astype(int)
    
    all_true.extend(y_test)
    all_pred.extend(preds)
    all_prob.extend(probs)
    all_vol.extend(volatile_test)
    
    elapsed = time.time() - start_time
    print(f"  Q{qi+1}: pred_up={preds.sum()}, pred_down={len(preds)-preds.sum()}, total={len(preds)} [{elapsed:.0f}s]")

ext_true = np.array(all_true)
ext_pred = np.array(all_pred)
ext_prob = np.array(all_prob)
ext_vol = np.array(all_vol)

# Full metrics
ext_metrics = evaluate_predictions(ext_true, ext_pred, ext_prob)
ext_vol_metrics = evaluate_predictions(ext_true[ext_vol==1], ext_pred[ext_vol==1], ext_prob[ext_vol==1])
ext_calm_metrics = evaluate_predictions(ext_true[ext_vol==0], ext_pred[ext_vol==0], ext_prob[ext_vol==0])

print(f"\n{'='*70}")
print("LSTM WITH LOG_RETURN RESULTS")
print(f"{'='*70}")

print(f"\n{'Regime':<15} {'Accuracy':>10} {'F1':>10} {'MCC':>10} {'AUC-ROC':>10}")
print("-" * 58)
print(f"{'Overall':<15} {ext_metrics['Accuracy']:>10.4f} {ext_metrics['F1 Score']:>10.4f} {ext_metrics['MCC']:>10.4f} {ext_metrics['AUC-ROC']:>10.4f}")
print(f"{'Volatile':<15} {ext_vol_metrics['Accuracy']:>10.4f} {ext_vol_metrics['F1 Score']:>10.4f} {ext_vol_metrics['MCC']:>10.4f} {ext_vol_metrics['AUC-ROC']:>10.4f}")
print(f"{'Calm':<15} {ext_calm_metrics['Accuracy']:>10.4f} {ext_calm_metrics['F1 Score']:>10.4f} {ext_calm_metrics['MCC']:>10.4f} {ext_calm_metrics['AUC-ROC']:>10.4f}")

print(f"\nPrediction balance: up={ext_pred.sum():,} ({ext_pred.mean()*100:.1f}%), down={len(ext_pred)-ext_pred.sum():,} ({(1-ext_pred.mean())*100:.1f}%)")
print(f"Baseline: 0.5141")

# Compare with previous LSTM-C
print(f"\n--- LSTM Comparison ---")
print(f"{'Model':<35} {'Accuracy':>10} {'MCC':>10}")
print("-" * 58)
print(f"{'LSTM (sentiment + volatility)':<35} {lstm_all_metrics['C_combined']['overall']['Accuracy']:>10.4f} {lstm_all_metrics['C_combined']['overall']['MCC']:>10.4f}")
print(f"{'LSTM (sent + vol + log_return)':<35} {ext_metrics['Accuracy']:>10.4f} {ext_metrics['MCC']:>10.4f}")
print(f"\nDoes adding price data help? {'Yes' if ext_metrics['MCC'] > lstm_all_metrics['C_combined']['overall']['MCC'] + 0.01 else 'No meaningful improvement'}")

print("\n=== OVERNIGHT RUN COMPLETE ===")

Building sequences with log_return...
Shape: (563213, 30, 3)

Running LSTM with ['sentiment', 'volatility_20d', 'log_return']
Parameters: hidden=32, epochs=15, seq_len=30

  Q1: pred_up=18483, pred_down=8155, total=26638 [54s]
  Q2: pred_up=15111, pred_down=10938, total=26049 [114s]
  Q3: pred_up=12029, pred_down=14381, total=26410 [183s]
  Q4: pred_up=14196, pred_down=11689, total=25885 [262s]
  Q5: pred_up=14043, pred_down=11914, total=25957 [348s]
  Q6: pred_up=14576, pred_down=11321, total=25897 [444s]
  Q7: pred_up=14610, pred_down=11857, total=26467 [553s]
  Q8: pred_up=8916, pred_down=16913, total=25829 [671s]
  Q9: pred_up=16235, pred_down=9253, total=25488 [805s]
  Q10: pred_up=14696, pred_down=11143, total=25839 [939s]
  Q11: pred_up=17599, pred_down=8902, total=26501 [1086s]
  Q12: pred_up=17282, pred_down=9473, total=26755 [1257s]
  Q13: pred_up=13966, pred_down=10113, total=24079 [1423s]
  Q14: pred_up=11407, pred_down=13132, total=24539 [1596s]
  Q15: pred_up=17054, pred_

## LSTM Interpretation

Adding log_return (price history) to the LSTM did not improve prediction. MCC remained near zero across all configurations. The prediction balance improved (56/44 vs 62/38 for the combined model without log_return), indicating the model was making more genuine predictions, but those predictions were not accurate.

The LSTM results reinforce the conclusion from logistic regression and XGBoost: the features do not contain sufficient signal for daily direction prediction, regardless of whether the model processes them as single-day inputs (logistic regression, XGBoost) or as 20-30 day sequences (LSTM).

## Overall Prediction Findings

Five models were tested across three architectures:

| Model | Accuracy | MCC | Approach |
|-------|----------|-----|----------|
| Logistic Regression | 50.4% | 0.005 | Linear, pooled, monthly walk-forward |
| XGBoost Per-Stock | 50.1% | 0.001 | Non-linear, per-stock, monthly walk-forward |
| LSTM Sentiment Only | 50.3% | -0.002 | Sequence, pooled, quarterly walk-forward |
| LSTM Combined | 50.4% | 0.001 | Sequence, pooled, quarterly walk-forward |
| LSTM + Log Return | 50.0% | -0.004 | Sequence with price, pooled, quarterly walk-forward |

All models produce accuracy near 50% and MCC near zero. The volatile vs calm accuracy gap is negligible across all models (< 0.5 percentage points). This confirms that the statistically significant moderation effect identified in Notebook 07 does not translate into practical daily prediction accuracy.

This finding is explained by the sign-flipping discovery: the interaction coefficient reverses direction between sub-periods, meaning any model trained on historical data learns the wrong relationship for future periods. Combined with Bloomberg's end-of-day sentiment aggregation — where much of the information is already reflected in prices by the time the score is available — daily direction prediction from these features is not feasible.